
# 📊 NB07 — Dashboard Dataset
## Investor Intelligence Platform — FIIs Brasil 🇧🇷
### CRISP-DM Phase: Deployment

| | |
|---|---|
| **Inputs** | Todos os outputs de NB02–NB06 |
| **Outputs** | `data/gold/dashboard/` — 5 datasets Parquet + 5 CSV + validações Plotly |
| **Engine** | PySpark + Pandas + Plotly |

## 🎯 O que este notebook faz

1. **Consolida** todos os Gold outputs em datasets prontos para consumo.
2. **Modela** 5 tabelas dashboard normalizadas:
   - `dashboard_articles` — catálogo de artigos com todos os scores
   - `dashboard_fii_signals` — métricas por FII com tendência
   - `dashboard_source_stats` — estatísticas por fonte
   - `dashboard_funnel_stats` — volumetria por estágio TOFU/MOFU/BOFU
   - `dashboard_word_cloud` — top 200 termos para nuvem de palavras
3. **Exporta** Parquet + CSV + JSON para FastAPI/Streamlit.
4. **Valida** com 4 gráficos Plotly interativos.

## 🗂️ Fluxo de Consolidação

```
NB02 Silver    ──────────────────────────────────────────┐
NB03 WordCount ─────────────────────────────────────────┐│
NB04 TF-IDF/BM25 ──────────────────────────────────────┐││
NB05 Sentiment ────────────────────────────────────────┐│││
NB06 MI Signals ──────────────────────────────────────┐││││
                                                       │││││
                         NB07 CONSOLIDATOR ───────────►│││││
                                                       └┘└┘└
                              ▼
              data/gold/dashboard/
                  dashboard_articles.parquet + .csv
                  dashboard_fii_signals.parquet + .csv
                  dashboard_source_stats.parquet + .csv
                  dashboard_funnel_stats.parquet + .csv
                  dashboard_word_cloud.parquet + .csv
                  api_payload_summary.json
```

## 📦 Seção 1 — Imports

In [1]:

import sys, os, json, warnings
os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
import re, random, unicodedata
from pathlib import Path
from datetime import datetime, timezone
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import pyarrow as pa

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pyspark

try:
    import plotly.graph_objects as go
    import plotly.express as px
    from plotly.subplots import make_subplots
    PLOTLY_OK = True
except ImportError:
    PLOTLY_OK = False

warnings.filterwarnings("ignore")
print(f"Python  : {sys.version.split()[0]}")
print(f"PySpark : {pyspark.__version__}")
print(f"Plotly  : {'OK' if PLOTLY_OK else 'NOT AVAILABLE'}")

Python  : 3.12.12
PySpark : 4.1.2
Plotly  : OK


## ⚙️ Seção 2 — Configuração e SparkSession

In [2]:

PROJECT_ROOT = Path(os.getcwd()).resolve()
_lim = 6
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT.parent != PROJECT_ROOT and _lim > 0:
    PROJECT_ROOT = PROJECT_ROOT.parent; _lim -= 1
sys.path.insert(0, str(PROJECT_ROOT))

from config.settings import (
    RANDOM_SEED, SPARK_DRIVER_MEMORY, SPARK_APP_NAME, SPARK_SHUFFLE_PARTS,
)
from src.utils.logger import ingestion_logger, log_quality_check, log_spark_start

os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

SILVER_DIR  = PROJECT_ROOT / "data" / "silver"
GOLD_WC     = PROJECT_ROOT / "data" / "gold" / "word_count"
GOLD_IDX    = PROJECT_ROOT / "data" / "gold" / "tfidf_bm25"
GOLD_SENT   = PROJECT_ROOT / "data" / "gold" / "sentiment"
GOLD_MI     = PROJECT_ROOT / "data" / "gold" / "marketing_intelligence"
GOLD_DASH   = PROJECT_ROOT / "data" / "gold" / "dashboard"
GOLD_DASH.mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName(f"{SPARK_APP_NAME}_dashboard")
    .master("local[*]")
    .config("spark.driver.memory",          SPARK_DRIVER_MEMORY)
    .config("spark.sql.shuffle.partitions", SPARK_SHUFFLE_PARTS)
    .config("spark.sql.adaptive.enabled",   "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

logger = ingestion_logger()
# FIXED: log_spark_start requer 3 argumentos (logger, app_name, spark_version)
log_spark_start(logger, spark.sparkContext.appName, spark.version)

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"GOLD_DASH    : {GOLD_DASH}")
print(f"Spark        : {spark.sparkContext.appName} | {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/14 15:55:08 WARN Utils: Your hostname, MacBook-Air-100.local, resolves to a loopback address: 127.0.0.1; using 192.168.15.4 instead (on interface en0)
26/06/14 15:55:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/14 15:55:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/14 15:55:09 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/14 15:55:09 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/06/14 15:55:09 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/06/14 15:55:09 WARN 

2026-06-14T15:55:11 | INFO     | fii_pipeline.ingestion | SPARK_START | app=FIIIntelligencePlatform_dashboard | spark=4.1.2
PROJECT_ROOT : /Users/fabicampanari/Desktop/project-fii-marketing-intelligence-platform
GOLD_DASH    : /Users/fabicampanari/Desktop/project-fii-marketing-intelligence-platform/data/gold/dashboard
Spark        : FIIIntelligencePlatform_dashboard | 4.1.2


## 📂 Seção 3 — Carregamento Resiliente de Todos os Inputs

In [3]:

def _load(paths, label, required=True):
    for p in (paths if isinstance(paths, list) else [paths]):
        p = Path(p)
        if p.exists():
            df = pd.read_parquet(p)
            print(f"  ✅ {label:<36} {p.name} — {len(df):,} linhas")
            return df
    if required:
        raise FileNotFoundError(f"{label} não encontrado. Execute NB correspondente.")
    print(f"  ⚠️  {label:<36} não encontrado — usando DataFrame vazio")
    return pd.DataFrame()

print("📂 Carregando todos os inputs Gold...")
df_silver = _load([
    SILVER_DIR / "silver_enriched.parquet",
    SILVER_DIR / "silver_articles.parquet",
], "Silver Enriched")

df_sent = _load([
    GOLD_SENT / "articles_sentiment.parquet",
], "Articles Sentiment", required=False)

df_sent_src = _load([
    GOLD_SENT / "sentiment_by_source.parquet",
], "Sentiment by Source", required=False)

df_sent_month = _load([
    GOLD_SENT / "sentiment_by_month.parquet",
], "Sentiment by Month", required=False)

df_wc_global = _load([
    GOLD_WC / "global_word_count.parquet",
], "Global Word Count", required=False)

df_tofu_freq = _load([
    GOLD_WC / "tofu_frequency.parquet",
], "TOFU Frequency", required=False)

df_mi_signals = _load([
    GOLD_MI / "mi_signals.parquet",
], "MI Signals", required=False)

df_mi_top = _load([
    GOLD_MI / "mi_top_articles.parquet",
], "MI Top Articles", required=False)

df_mi_funnel = _load([
    GOLD_MI / "mi_funnel.parquet",
], "MI Funnel", required=False)

df_doc_map = _load([
    GOLD_IDX / "doc_index_map.parquet",
], "Doc Index Map", required=False)

print(f"\n✅ Todos os inputs carregados com sucesso")

📂 Carregando todos os inputs Gold...
  ✅ Silver Enriched                      silver_enriched.parquet — 126 linhas
  ✅ Articles Sentiment                   articles_sentiment.parquet — 126 linhas
  ✅ Sentiment by Source                  sentiment_by_source.parquet — 16 linhas
  ✅ Sentiment by Month                   sentiment_by_month.parquet — 4 linhas
  ✅ Global Word Count                    global_word_count.parquet — 7,752 linhas
  ✅ TOFU Frequency                       tofu_frequency.parquet — 29 linhas
  ✅ MI Signals                           mi_signals.parquet — 15 linhas
  ✅ MI Top Articles                      mi_top_articles.parquet — 150 linhas
  ✅ MI Funnel                            mi_funnel.parquet — 300 linhas
  ✅ Doc Index Map                        doc_index_map.parquet — 126 linhas

✅ Todos os inputs carregados com sucesso



## 🏗️ Seção 4 — `dashboard_articles` (Tabela Principal)

Catálogo completo de artigos com todos os scores consolidados.
Esta é a tabela central que a FastAPI e o Streamlit consultam.

In [4]:
print("\n🏗️ Construindo dashboard_articles...")

df_base = df_silver.copy()
_text_col = "text_clean"
df_base[_text_col] = (
    df_base.get(_text_col, pd.Series([""] * len(df_base)))
    .fillna("")
)

# ── Mesclar scores de sentimento se não estiverem no Silver ──
_sent_cols = [
    "polarity_score", "sentiment_label", "n_pos_terms", "n_neg_terms",
    "flag_dividendo", "flag_oportunidade", "flag_risco",
    "flag_crise", "flag_vacancia", "flag_inadimplencia",
    "score_dividendo", "score_oportunidade", "score_risco",
    "score_crise"
]
_missing = [c for c in _sent_cols if c not in df_base.columns]
if _missing and not df_sent.empty:
    _cols_avail = ["article_id"] + [
        c for c in _missing if c in df_sent.columns
    ]
    df_base = df_base.merge(
        df_sent[_cols_avail], on="article_id", how="left"
    )

# ── Mesclar scores MI se disponíveis ────────────────────────
if not df_mi_top.empty:
    _mi_agg = (
        df_mi_top.groupby("article_id")
        .agg(
            score_bm25=("score_bm25", "max"),
            score_tfidf=("score_tfidf", "max"),
            score_hybrid=("score_hybrid", "max"),
            mi_article_score=("mi_article_score", "max"),
        )
        .reset_index()
    )
    df_base = df_base.merge(
        _mi_agg, on="article_id", how="left"
    )

# ── Selecionar + renomear colunas finais ───────────────────
_src_lbl = (
    "source_label" if "source_label" in df_base.columns else "source"
)
_final_cols = [
    "article_id", "source", "source_type", _src_lbl,
    "title", "url", "author", "published_at", "published_dt",
    "collected_at", "language", "word_count", "char_count",
    "has_content", "polarity_score", "sentiment_label",
    "n_pos_terms", "n_neg_terms", "flag_dividendo",
    "flag_oportunidade", "flag_risco", "flag_crise",
    "flag_vacancia", "flag_inadimplencia", "score_dividendo",
    "score_oportunidade", "score_risco", "score_crise",
    "score_bm25", "score_tfidf", "score_hybrid",
    "mi_article_score", "content_hash", "query_used",
    "ingestion_method",
]
_avail = [c for c in _final_cols if c in df_base.columns]
_avail = list(dict.fromkeys(_avail))  # dedup preservando ordem

df_dash_articles = (
    df_base[_avail]
    .drop_duplicates("article_id")
    .reset_index(drop=True)
)

# Preencher NaN em scores
for c in [
    "polarity_score", "score_bm25", "score_tfidf",
    "score_hybrid", "mi_article_score"
]:
    if c in df_dash_articles.columns:
        df_dash_articles[c] = df_dash_articles[c].fillna(0.0)
for c in ["sentiment_label"]:
    if c in df_dash_articles.columns:
        df_dash_articles[c] = df_dash_articles[c].fillna("neutro")
for c in [
    "flag_dividendo", "flag_oportunidade", "flag_risco",
    "flag_crise", "flag_vacancia", "flag_inadimplencia"
]:
    if c in df_dash_articles.columns:
        df_dash_articles[c] = (
            df_dash_articles[c].fillna(False).astype(bool)
        )

_p = GOLD_DASH / "dashboard_articles.parquet"
df_dash_articles.to_parquet(_p, index=False, compression="snappy")
df_dash_articles.to_csv(
    GOLD_DASH / "dashboard_articles.csv",
    index=False,
    encoding="utf-8"
)
print(
    f"✅ dashboard_articles: {len(df_dash_articles):,} artigos × "
    f"{len(df_dash_articles.columns)} cols"
)


🏗️ Construindo dashboard_articles...
✅ dashboard_articles: 126 artigos × 35 cols


## 📈 Seção 5 — `dashboard_fii_signals` (Métricas por FII)

In [5]:

print("\n📈 Construindo dashboard_fii_signals...")

if not df_mi_signals.empty:
    df_fii_signals = df_mi_signals.copy()
else:
    # Fallback: gerar a partir de mentions nos artigos
    _text_col = "text_clean" if "text_clean" in df_silver.columns else "title"
    FII_TICKERS = ["HGLG11","KNRI11","XPML11","CSHG11","VISC11",
                   "BRCO11","BTLG11","MXRF11","HCRI11","ALZR11",
                   "BRCR11","HSML11","VGIP11","GGRC11","RBVA11"]
    rows = []
    for tkr in FII_TICKERS:
        mask = df_silver[_text_col].str.contains(tkr, na=False, case=False)
        sub  = df_silver[mask]
        rows.append({
            "ticker": tkr, "mentions": len(sub),
            "sentiment_avg": sub.get("polarity_score", pd.Series([0.0]*len(sub))).mean(),
            "risk_score": 0.0, "opportunity_score": 0.0, "mi_score": 0.0,
        })
    df_fii_signals = pd.DataFrame(rows)

# Adicionar tendência de sentimento (normalizada 0-1)
_mn = df_fii_signals["sentiment_avg"].min()
_mx = df_fii_signals["sentiment_avg"].max()
df_fii_signals["sentiment_trend"] = (
    (df_fii_signals["sentiment_avg"] - _mn) / (_mx - _mn + 1e-9)
).round(4) if _mx != _mn else 0.5

_p = GOLD_DASH / "dashboard_fii_signals.parquet"
df_fii_signals.to_parquet(_p, index=False, compression="snappy")
df_fii_signals.to_csv(GOLD_DASH / "dashboard_fii_signals.csv", index=False, encoding="utf-8")
print(f"✅ dashboard_fii_signals: {len(df_fii_signals)} FIIs × {len(df_fii_signals.columns)} cols")
print(df_fii_signals[["ticker","mentions","sentiment_avg","risk_score","mi_score"]].to_string(index=False))


📈 Construindo dashboard_fii_signals...
✅ dashboard_fii_signals: 15 FIIs × 18 cols
ticker  mentions  sentiment_avg  risk_score  mi_score
HCRI11        90         0.2719      1.2222    0.4320
XPML11        75         0.2759      1.1600    0.3959
VGIP11        86         0.2596      1.2326    0.3954
HSML11        72         0.2140      1.2778    0.3887
MXRF11       102         0.2657      1.1765    0.3834
KNRI11        83         0.2791      1.3253    0.3759
VISC11        68         0.2376      1.2794    0.3640
ALZR11        85         0.2688      1.2941    0.3628
BTLG11        73         0.2529      1.2329    0.3498
BRCR11        67         0.2470      1.2239    0.3482
HGLG11        73         0.2343      1.3425    0.3465
RBVA11        91         0.1729      1.4176    0.3432
BRCO11        71         0.2449      1.2676    0.3351
CSHG11        65         0.2300      1.4615    0.3309
GGRC11        66         0.2498      1.2273    0.3204


## 📡 Seção 6 — `dashboard_source_stats` (Estatísticas por Fonte)

In [6]:
print("\n📡 Construindo dashboard_source_stats...")

_src_col = (
    "source_label" if "source_label" in df_dash_articles.columns
    else "source"
)

sdf_articles = spark.createDataFrame(
    df_dash_articles.fillna("").fillna(0.0).fillna(False)
)

sdf_src_stats = (
    sdf_articles
    .groupBy(_src_col)
    .agg(
        F.count("article_id").alias("total_articles"),
        F.avg("polarity_score").alias("avg_polarity"),
        F.sum(
            F.when(F.col("sentiment_label") == "positivo", 1)
            .otherwise(0)
        ).alias("n_positivo"),
        F.sum(
            F.when(F.col("sentiment_label") == "negativo", 1)
            .otherwise(0)
        ).alias("n_negativo"),
        F.sum(
            F.when(F.col("sentiment_label") == "neutro", 1)
            .otherwise(0)
        ).alias("n_neutro"),
        F.sum(F.col("flag_dividendo").cast("int")).alias("n_dividendo"),
        F.sum(F.col("flag_risco").cast("int")).alias("n_risco"),
        F.sum(F.col("flag_crise").cast("int")).alias("n_crise"),
        F.avg("word_count").alias("avg_word_count"),
        F.first("source_type").alias("source_type"),
    )
    .withColumn(
        "pct_positivo",
        F.round(
            F.col("n_positivo") / F.col("total_articles") * 100, 2
        )
    )
    .withColumn(
        "pct_negativo",
        F.round(
            F.col("n_negativo") / F.col("total_articles") * 100, 2
        )
    )
    .withColumn("avg_polarity", F.round("avg_polarity", 4))
    .withColumn("avg_word_count", F.round("avg_word_count", 1))
    .orderBy(F.col("total_articles").desc())
)

df_src_stats = sdf_src_stats.toPandas()
_p = GOLD_DASH / "dashboard_source_stats.parquet"
df_src_stats.to_parquet(_p, index=False, compression="snappy")
df_src_stats.to_csv(
    GOLD_DASH / "dashboard_source_stats.csv",
    index=False,
    encoding="utf-8"
)
print(
    f"✅ dashboard_source_stats: {len(df_src_stats)} fontes × "
    f"{len(df_src_stats.columns)} cols"
)
print(
    df_src_stats[
        [_src_col, "total_articles", "avg_polarity", "pct_positivo",
         "pct_negativo"]
    ].to_string(index=False)
)


📡 Construindo dashboard_source_stats...


26/06/14 15:55:13 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


✅ dashboard_source_stats: 16 fontes × 13 cols
        source_label  total_articles  avg_polarity  pct_positivo  pct_negativo
           Empiricus              32        0.2746         59.38          9.38
 CNN Brasil Business              17       -0.2633         17.65         58.82
      Funds Explorer              12        0.3473         75.00          0.00
         FIIs.com.br              10        0.6113         90.00          0.00
        Investidor10              10        0.6470        100.00          0.00
       Status Invest               8        0.4541         62.50          0.00
   Eu Quero Investir               7        0.3323         57.14          0.00
        Seu Dinheiro               6        0.0464         16.67         16.67
       Suno Research               6        0.0928         16.67         16.67
  Bora Investir (B3)               4        0.3465         75.00          0.00
         Money Times               4        0.1100         50.00         25.00
      

## 🔄 Seção 7 — `dashboard_funnel_stats` + `dashboard_word_cloud`

In [7]:
# ── Funil TOFU/MOFU/BOFU ─────────────────────────────────────────────────────
print("\n🔄 Construindo dashboard_funnel_stats...")
if not df_mi_funnel.empty:
    df_funil_stats = (
        df_mi_funnel.groupby("stage")
        .agg(
            n_queries=("query", "nunique"),
            n_docs=("article_id", "count"),
            avg_score=("score_hybrid", "mean"),
            avg_polarity=("polarity_score", "mean"),
            n_positivo=(
                "sentiment_label",
                lambda x: (x == "positivo").sum()
            ),
            n_negativo=(
                "sentiment_label",
                lambda x: (x == "negativo").sum()
            ),
            n_dividendo=("flag_dividendo", "sum"),
            n_risco=("flag_risco", "sum"),
        )
        .round(4)
        .reset_index()
    )
else:
    df_funil_stats = pd.DataFrame({
        "stage": ["TOFU", "MOFU", "BOFU"],
        "n_docs": [0, 0, 0],
        "avg_score": [0.0, 0.0, 0.0],
    })

_p = GOLD_DASH / "dashboard_funnel_stats.parquet"
df_funil_stats.to_parquet(_p, index=False, compression="snappy")
df_funil_stats.to_csv(
    GOLD_DASH / "dashboard_funnel_stats.csv",
    index=False,
    encoding="utf-8"
)
print(f"✅ dashboard_funnel_stats: {len(df_funil_stats)} estágios")
print(df_funil_stats.to_string(index=False))

# ── Word Cloud (top 200 termos) ───────────────────────────────────────────────
print("\n☁️  Construindo dashboard_word_cloud...")
if not df_wc_global.empty:
    df_wc_top = df_wc_global.nlargest(200, "count").reset_index(drop=True)
    df_wc_top["freq_normalized"] = (
        df_wc_top["count"] / df_wc_top["count"].max()
    ).round(4)
    # Enriquecer com info TOFU
    if not df_tofu_freq.empty:
        _tofu_set = set(df_tofu_freq["term"].tolist())
        df_wc_top["is_tofu"] = df_wc_top["term"].isin(_tofu_set)
    else:
        df_wc_top["is_tofu"] = False
else:
    df_wc_top = pd.DataFrame(
        {"term": [], "count": [], "freq_normalized": [], "is_tofu": []}
    )

_p = GOLD_DASH / "dashboard_word_cloud.parquet"
df_wc_top.to_parquet(_p, index=False, compression="snappy")
df_wc_top.to_csv(
    GOLD_DASH / "dashboard_word_cloud.csv",
    index=False,
    encoding="utf-8"
)
print(f"✅ dashboard_word_cloud: {len(df_wc_top)} termos")
if not df_wc_top.empty:
    print(f"   Top 10: {df_wc_top.head(10)['term'].tolist()}")


🔄 Construindo dashboard_funnel_stats...
✅ dashboard_funnel_stats: 3 estágios
stage  n_queries  n_docs  avg_score  avg_polarity  n_positivo  n_negativo  n_dividendo  n_risco
 BOFU          5     100     0.5328        0.3067          59           5           64       45
 MOFU          5     100     0.4569        0.2752          49           5           59       38
 TOFU          5     100     0.5108        0.4198          66           2           66       28

☁️  Construindo dashboard_word_cloud...
✅ dashboard_word_cloud: 200 termos
   Top 10: ['mercado', 'nao', 'fundo', 'carteira', 'fundos', 'imagem', 'renda', 'trocar', 'ativos', 'dividendos']


## 🔌 Seção 8 — JSON Summary para FastAPI

In [8]:

print("\n🔌 Gerando api_payload_summary.json...")

_lbl_col = "source_label" if "source_label" in df_src_stats.columns else _src_col
summary = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "pipeline_version": "1.0.0",
    "random_seed": RANDOM_SEED,
    "totals": {
        "articles":    int(len(df_dash_articles)),
        "sources":     int(df_dash_articles["source"].nunique()),
        "fii_entities": int(len(df_fii_signals)),
    },
    "sentiment_distribution": df_dash_articles.get(
        "sentiment_label", pd.Series(dtype=str)
    ).value_counts().to_dict() if "sentiment_label" in df_dash_articles.columns else {},
    "top_sources": (
        df_src_stats[[_lbl_col,"total_articles"]]
        .sort_values("total_articles", ascending=False)
        .head(5)
        .to_dict(orient="records")
    ) if not df_src_stats.empty else [],
    "top_fii_by_mi_score": (
        df_fii_signals[["ticker","mi_score","sentiment_avg","mentions"]]
        .sort_values("mi_score", ascending=False)
        .head(5)
        .to_dict(orient="records")
    ) if "mi_score" in df_fii_signals.columns else [],
    "source_types": df_dash_articles.get(
        "source_type", pd.Series(dtype=str)
    ).value_counts().to_dict() if "source_type" in df_dash_articles.columns else {},
    "data_paths": {
        "dashboard_articles":   str(GOLD_DASH / "dashboard_articles.parquet"),
        "dashboard_fii_signals":str(GOLD_DASH / "dashboard_fii_signals.parquet"),
        "dashboard_source_stats":str(GOLD_DASH / "dashboard_source_stats.parquet"),
        "dashboard_funnel_stats":str(GOLD_DASH / "dashboard_funnel_stats.parquet"),
        "dashboard_word_cloud": str(GOLD_DASH / "dashboard_word_cloud.parquet"),
    },
}

_json_path = GOLD_DASH / "api_payload_summary.json"
_json_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False, default=str))
print(f"✅ api_payload_summary.json salvo")
print(f"   Artigos  : {summary['totals']['articles']:,}")
print(f"   Fontes   : {summary['totals']['sources']}")
print(f"   FIIs     : {summary['totals']['fii_entities']}")
if summary["sentiment_distribution"]:
    print(f"   Sentimento: {summary['sentiment_distribution']}")


🔌 Gerando api_payload_summary.json...
✅ api_payload_summary.json salvo
   Artigos  : 126
   Fontes   : 16
   FIIs     : 15
   Sentimento: {'positivo': 70, 'neutro': 39, 'negativo': 17}


## 📊 Seção 9 — Visualizações de Validação Plotly

In [9]:
if not PLOTLY_OK:
    print("⚠️  Plotly não instalado — pip install plotly")
else:
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[
            "Artigos por Fonte (Top 12)",
            "Distribuição de Sentimento",
            "MI Score por FII",
            "Funil de Marketing TOFU/MOFU/BOFU",
        ],
        specs=[
            [{"type": "xy"}, {"type": "domain"}],
            [{"type": "xy"}, {"type": "xy"}],
        ],
        vertical_spacing=0.14,
        horizontal_spacing=0.1,
    )

    # ── P1: Artigos por fonte ────────────────────────────────────
    _lc = ("source_label" if "source_label" in df_src_stats.columns
           else "source")
    _top12 = df_src_stats.nlargest(12, "total_articles")
    fig.add_trace(go.Bar(
        x=_top12[_lc].tolist(),
        y=_top12["total_articles"].tolist(),
        marker_color="#00d2ff",
        name="Artigos",
        showlegend=False,
    ), row=1, col=1)

    # ── P2: Sentimento ──────────────────────────────────────────
    if "sentiment_label" in df_dash_articles.columns:
        _sd = df_dash_articles["sentiment_label"].value_counts()
        _colors_sent = {
            "positivo": "#2ecc71",
            "neutro": "#95a5a6",
            "negativo": "#e74c3c"
        }
        fig.add_trace(go.Pie(
            labels=_sd.index.tolist(),
            values=_sd.values.tolist(),
            marker_colors=[
                _colors_sent.get(l, "#7f8c8d") for l in _sd.index
            ],
            showlegend=True,
            hole=0.4,
        ), row=1, col=2)

    # ── P3: MI Score por FII ────────────────────────────────────
    if (not df_fii_signals.empty and
            "mi_score" in df_fii_signals.columns):
        _fii_top = df_fii_signals.nlargest(10, "mi_score")
        _sent_avg = _fii_top.get(
            "sentiment_avg",
            pd.Series([0] * len(_fii_top))
        )
        _bar_colors = [
            "#2ecc71" if v >= 0 else "#e74c3c" for v in _sent_avg
        ]
        fig.add_trace(go.Bar(
            x=_fii_top["ticker"].tolist(),
            y=_fii_top["mi_score"].tolist(),
            marker_color=_bar_colors,
            name="MI Score",
            showlegend=False,
        ), row=2, col=1)

    # ── P4: Funil TOFU/MOFU/BOFU ────────────────────────────────
    if "n_docs" in df_funil_stats.columns:
        _stage_colors = {
            "TOFU": "#00d2ff",
            "MOFU": "#a8e6cf",
            "BOFU": "#ff6b6b"
        }
        fig.add_trace(go.Bar(
            x=df_funil_stats["stage"].tolist(),
            y=df_funil_stats["n_docs"].tolist(),
            marker_color=[
                _stage_colors.get(s, "#7f8c8d")
                for s in df_funil_stats["stage"]
            ],
            name="Docs por Estágio",
            showlegend=False,
        ), row=2, col=2)

    fig.update_layout(
        title_text=(
            "NB07 — Dashboard Validation: Investor Intelligence "
            "Platform FIIs 🇧🇷"
        ),
        template="plotly_dark",
        height=700,
        paper_bgcolor="#0d1117",
        plot_bgcolor="#161b22",
        font=dict(color="#c9d1d9", size=11),
        title_font=dict(size=16, color="#00d2ff"),
    )
    fig.update_xaxes(tickangle=-30)
    html_path = GOLD_DASH / "dashboard_validation.html"
    fig.write_html(str(html_path))
    print(f"✅ dashboard_validation.html salvo")

✅ dashboard_validation.html salvo


## ✅ Seção 10 — Validação Final do Pipeline Completo

In [10]:

print("\n" + "=" * 70)
print("✅  VALIDAÇÃO FINAL — Investor Intelligence Platform FIIs Brasil 🇧🇷")
print("=" * 70)

PIPELINE_CHECKS = [
    ("NB00 config/settings.py",          (PROJECT_ROOT / "config" / "settings.py").exists()),
    ("NB00 src/utils/logger.py",         (PROJECT_ROOT / "src" / "utils" / "logger.py").exists()),
    ("NB01 Bronze (data/external/)",     any((PROJECT_ROOT/"data"/"external").glob("*.parquet"))
                                          if (PROJECT_ROOT/"data"/"external").exists() else False),
    ("NB02 Silver articles",             (SILVER_DIR/"silver_articles.parquet").exists()),
    ("NB02 Silver enriched",             (SILVER_DIR/"silver_enriched.parquet").exists()),
    ("NB03 Global word count",           (GOLD_WC/"global_word_count.parquet").exists()),
    ("NB03 TOFU frequency",              (GOLD_WC/"tofu_frequency.parquet").exists()),
    ("NB04 TF-IDF vectorizer",           (GOLD_IDX/"tfidf_vectorizer.pkl").exists()),
    ("NB04 BM25 index",                  (GOLD_IDX/"bm25_index.pkl").exists()),
    ("NB05 Articles sentiment",          (GOLD_SENT/"articles_sentiment.parquet").exists()),
    ("NB05 Sentiment by source",         (GOLD_SENT/"sentiment_by_source.parquet").exists()),
    ("NB06 MI Signals",                  (GOLD_MI/"mi_signals.parquet").exists()),
    ("NB06 MI Top Articles",             (GOLD_MI/"mi_top_articles.parquet").exists()),
    ("NB06 MI Funnel",                   (GOLD_MI/"mi_funnel.parquet").exists()),
    ("NB07 dashboard_articles",          (GOLD_DASH/"dashboard_articles.parquet").exists()),
    ("NB07 dashboard_fii_signals",       (GOLD_DASH/"dashboard_fii_signals.parquet").exists()),
    ("NB07 dashboard_source_stats",      (GOLD_DASH/"dashboard_source_stats.parquet").exists()),
    ("NB07 dashboard_funnel_stats",      (GOLD_DASH/"dashboard_funnel_stats.parquet").exists()),
    ("NB07 dashboard_word_cloud",        (GOLD_DASH/"dashboard_word_cloud.parquet").exists()),
    ("NB07 api_payload_summary.json",    (GOLD_DASH/"api_payload_summary.json").exists()),
]

passed = 0
for name, ok in PIPELINE_CHECKS:
    icon = "✅" if ok else "❌"
    print(f"  {icon} {name}")
    if ok: passed += 1

print(f"\n  {passed}/{len(PIPELINE_CHECKS)} checks passaram")

# ── Resumo de artefatos finais ────────────────────────────────────────────────
print("\n📊 Dashboard Datasets finais:")
for fname in [
    "dashboard_articles.parquet",
    "dashboard_fii_signals.parquet",
    "dashboard_source_stats.parquet",
    "dashboard_funnel_stats.parquet",
    "dashboard_word_cloud.parquet",
]:
    p = GOLD_DASH / fname
    if p.exists():
        df_ = pd.read_parquet(p)
        sz  = p.stat().st_size // 1024
        print(f"  {fname:<46} {len(df_):>6,} linhas | {sz:>5} KB")

log_quality_check(
    logger, nb="NB07",
    pipeline_checks_passed=passed,
    pipeline_checks_total=len(PIPELINE_CHECKS),
    dashboard_articles=len(df_dash_articles),
    fii_entities=len(df_fii_signals),
)

print("\n" + "=" * 70)
if passed == len(PIPELINE_CHECKS):
    print("🎉 PIPELINE COMPLETO — Todos os 8 notebooks executados com sucesso!")
else:
    print(f"⚠️  Execute os notebooks faltantes e re-rode NB07 para completar o pipeline.")
print("   FastAPI  → api/app.py      | uvicorn api.app:app --reload")
print("   Dashboard → dashboard/     | streamlit run dashboard/Home.py")
print("=" * 70)


✅  VALIDAÇÃO FINAL — Investor Intelligence Platform FIIs Brasil 🇧🇷
  ✅ NB00 config/settings.py
  ✅ NB00 src/utils/logger.py
  ✅ NB01 Bronze (data/external/)
  ✅ NB02 Silver articles
  ✅ NB02 Silver enriched
  ✅ NB03 Global word count
  ✅ NB03 TOFU frequency
  ✅ NB04 TF-IDF vectorizer
  ✅ NB04 BM25 index
  ✅ NB05 Articles sentiment
  ✅ NB05 Sentiment by source
  ✅ NB06 MI Signals
  ✅ NB06 MI Top Articles
  ✅ NB06 MI Funnel
  ✅ NB07 dashboard_articles
  ✅ NB07 dashboard_fii_signals
  ✅ NB07 dashboard_source_stats
  ✅ NB07 dashboard_funnel_stats
  ✅ NB07 dashboard_word_cloud
  ✅ NB07 api_payload_summary.json

  20/20 checks passaram

📊 Dashboard Datasets finais:
  dashboard_articles.parquet                        126 linhas |    58 KB
  dashboard_fii_signals.parquet                      15 linhas |    12 KB
  dashboard_source_stats.parquet                     16 linhas |     8 KB
  dashboard_funnel_stats.parquet                      3 linhas |     5 KB
  dashboard_word_cloud.parquet     


## 🗂️ Mapa Completo de Artefatos

| Notebook | Camada | Artefatos Principais |
|----------|--------|----------------------|
| NB00 | Setup | `config/settings.py` · `src/utils/logger.py` · stubs API/Dashboard |
| NB01 | Bronze | `data/external/bronze_all_articles.parquet` (21 fontes) |
| NB02 | Silver | `data/silver/silver_articles.parquet` |
| NB03 | Gold | `word_count/global_word_count.parquet` · `tofu_frequency.parquet` |
| NB04 | Gold | `tfidf_bm25/tfidf_vectorizer.pkl` · `bm25_index.pkl` |
| NB05 | Silver+Gold | `silver_enriched.parquet` · `sentiment/articles_sentiment.parquet` |
| NB06 | Gold | `mi_signals.parquet` · `mi_top_articles.parquet` · `mi_funnel.parquet` |
| **NB07** | **Gold** | **`dashboard/*.parquet` · `dashboard/*.csv` · `api_payload_summary.json`** |

## ✅ NB07 Completo — Pipeline FII 100% executado 🎉

```
Próximos passos:
  uvicorn api.app:app --reload
  streamlit run dashboard/Home.py
```